# API Data Wrangling with Open-Meteo

In this exercise, we will:

- Fetch historical weather data using the **Open-Meteo API**.
- Parse and clean the data using **pandas**.
- Perform exploratory data analysis.
- Visualize weather trends.

In [ ]:
# Install required packages if not already installed
# !pip install requests pandas matplotlib --quiet

In [ ]:
# Import Libraries
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Optional: Configure matplotlib for inline display in Jupyter
%matplotlib inline

## Understand the Open-Meteo API

- **API Endpoint**: `https://archive-api.open-meteo.com/v1/archive`
- **Parameters**:
  - `latitude`: Latitude of the location.
  - `longitude`: Longitude of the location.
  - `start_date`: Start date of the data (YYYY-MM-DD).
  - `end_date`: End date of the data (YYYY-MM-DD).
  - `hourly`: Comma-separated list of variables (e.g., temperature, precipitation).
  - `timezone`: Timezone of the data.

**Documentation**: [Open-Meteo API Docs](https://open-meteo.com/en/docs)

In [ ]:
# Step 1: Set Up Parameters

# Location coordinates for New York City
latitude = 40.7128
longitude = -74.0060

# Date range
start_date = '2023-01-01'
end_date = '2023-01-07'  # One week of data

# Variables to fetch
hourly_variables = ['temperature_2m', 'relativehumidity_2m', 'precipitation']


In [ ]:
# Step 2: Make the API Request

base_url = 'https://archive-api.open-meteo.com/v1/archive'

params = {
    'latitude': latitude,
    'longitude': longitude,
    'start_date': start_date,
    'end_date': end_date,
    'hourly': ','.join(hourly_variables),
    'timezone': 'America/New_York'
}

response = requests.get(base_url, params=params)


In [ ]:
# Check if the request was successful

if response.status_code == 200:
    print('Data fetched successfully!')
else:
    print(f'Failed to fetch data. Status code: {response.status_code}')


In [ ]:
# Step 3: Load Data into pandas DataFrame

data = response.json()
hourly_data = data['hourly']

df = pd.DataFrame(hourly_data)
df.head()


In [ ]:
# Step 4: Data Cleaning

# Convert 'time' column to datetime
df['time'] = pd.to_datetime(df['time'])

# Set 'time' as the index
df.set_index('time', inplace=True)

# Check for missing values
df.isnull().sum()


In [ ]:
# Handle missing values (if any)
df.fillna(method='ffill', inplace=True)  # Forward fill


In [ ]:
# Step 5: Exploratory Data Analysis

# Summary Statistics
df.describe()

In [ ]:
# Plot Temperature Over Time

plt.figure(figsize=(14, 6))
plt.plot(df.index, df['temperature_2m'], label='Temperature (°C)')
plt.title('Temperature Over Time in New York City')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.show()

In [ ]:
# Plot Relative Humidity Over Time

plt.figure(figsize=(14, 6))
plt.plot(df.index, df['relativehumidity_2m'], color='orange', label='Relative Humidity (%)')
plt.title('Relative Humidity Over Time in New York City')
plt.xlabel('Date')
plt.ylabel('Relative Humidity (%)')
plt.legend()
plt.show()


In [ ]:
# Plot Precipitation Over Time

plt.figure(figsize=(14, 6))
plt.bar(df.index, df['precipitation'], label='Precipitation (mm)')
plt.title('Precipitation Over Time in New York City')
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.legend()
plt.show()


In [ ]:
# Step 6: Correlation Analysis

# Calculate correlation between temperature and humidity
correlation = df['temperature_2m'].corr(df['relativehumidity_2m'])
print(f'Correlation between Temperature and Relative Humidity: {correlation:.2f}')


In [ ]:
# Scatter Plot of Temperature vs. Relative Humidity

plt.figure(figsize=(8, 6))
plt.scatter(df['temperature_2m'], df['relativehumidity_2m'], alpha=0.5)
plt.title('Temperature vs. Relative Humidity')
plt.xlabel('Temperature (°C)')
plt.ylabel('Relative Humidity (%)')
plt.show()


In [ ]:
# Step 7: Resample Data to Daily Averages

daily_avg = df.resample('D').mean()
daily_avg.head()


In [ ]:
# Plot Daily Average Temperature

plt.figure(figsize=(10, 5))
plt.plot(daily_avg.index, daily_avg['temperature_2m'], marker='o')
plt.title('Daily Average Temperature in New York City')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.show()


In [ ]:
# Step 8: Compare with Another Location (Los Angeles)

# Coordinates for Los Angeles
latitude_la = 34.0522
longitude_la = -118.2437

params_la = {
    'latitude': latitude_la,
    'longitude': longitude_la,
    'start_date': start_date,
    'end_date': end_date,
    'hourly': ','.join(hourly_variables),
    'timezone': 'America/Los_Angeles'
}

response_la = requests.get(base_url, params=params_la)

# Load and clean Los Angeles data
data_la = response_la.json()
hourly_data_la = data_la['hourly']
df_la = pd.DataFrame(hourly_data_la)

df_la['time'] = pd.to_datetime(df_la['time'])
df_la.set_index('time', inplace=True)
df_la.fillna(method='ffill', inplace=True)


In [ ]:
# Resample Los Angeles Data to Daily Averages

daily_avg_la = df_la.resample('D').mean()


In [ ]:
# Combine DataFrames for Comparison

combined_temp = pd.DataFrame({
    'New York': daily_avg['temperature_2m'],
    'Los Angeles': daily_avg_la['temperature_2m']
})


In [ ]:
# Plot Comparison of Daily Average Temperature

combined_temp.plot(kind='bar', figsize=(10, 6))
plt.title('Daily Average Temperature: New York vs Los Angeles')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


In [ ]:
# Step 9: Advanced Analysis - Temperature Difference

# Calculate Temperature Difference
combined_temp['Temp Difference'] = combined_temp['Los Angeles'] - combined_temp['New York']
combined_temp


In [ ]:
# Step 10: Save Data to CSV (Optional)

# Save New York data
df.to_csv('new_york_weather.csv')

# Save Los Angeles data
df_la.to_csv('los_angeles_weather.csv')


## Conclusion

In this exercise, we:

- Accessed and retrieved data from the **Open-Meteo API**.
- Cleaned and preprocessed JSON data using **pandas**.
- Performed exploratory data analysis and visualization.
- Compared weather patterns between **New York City** and **Los Angeles**.

---

**Next Steps**:

- **Extend the Date Range**: Analyze seasonal trends by fetching data over several months.
- **Fetch Additional Variables**: Include wind speed, atmospheric pressure, or other interesting variables.
- **Forecasting**: Use statistical methods or machine learning models to forecast future weather patterns.
- **Global Analysis**: Compare data from other global cities to gain broader insights.


## Exercises for Students

Now it's your turn! Below are some exercises to reinforce what you've learned. Try to complete them without looking at the solutions.

### Exercise: Compare Rainfall Between London and Seattle

- **Objective**: Compare the total weekly rainfall between London and Seattle over a full year.
- **Tasks**:
  1. **Fetch Data**: Retrieve hourly precipitation data for London and Seattle for the year 2022.
     - **London Coordinates**: Latitude `51.5074`, Longitude `-0.1278`
     - **Seattle Coordinates**: Latitude `47.6062`, Longitude `-122.3321`
  2. **Data Cleaning**: Convert the time columns to datetime objects and set them as the index. Handle any missing values.
  3. **Resample Data**: Resample the hourly data to weekly totals.
  4. **Visualization**: Plot the weekly total precipitation for both cities on the same graph.
  5. **Analysis**: Determine which city had more rainfall overall and identify any interesting patterns.

**Note**: Remember to handle any API limitations, such as data availability or rate limits, and to be mindful of the size of the data you're requesting.

### Tips:

- **API Parameters**: Make sure to adjust the parameters like `latitude`, `longitude`, `start_date`, `end_date`, `hourly`, and `timezone` as needed.
- **Error Handling**: Always check if your API requests are successful before proceeding.
- **Data Storage**: Consider saving your DataFrames to CSV files for future analysis.
- **Visualization**: Customize your plots with titles, labels, legends, and gridlines for better readability.

Happy coding!


In [ ]:
# Import Libraries
import requests
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
#Setting up the parameters
base_url = "https://archive-api.open-meteo.com/v1/archive"
locations = {
    "London": {"lat": 51.5074, "lon": -0.1278},
    "Seattle": {"lat": 47.6062, "lon": -122.3321}
}
start_date = "2022-01-01"
end_date = "2022-12-31"

hourly_variables = [
    "precipitation", 
    "wind_speed_10m", 
    "surface_pressure",
    "temperature_2m"
]
daily_variables = ["sunrise", "sunset"]

In [ ]:
#fetching the data
# Create a dictionary to store the results for each city
weather_data = {}

for city, coords in locations.items():
    print(f"Fetching data for {city}...")
    
    # Define the parameters for THIS specific city
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "daily": ",".join(daily_variables),
        "timezone": "auto" # Automatically adjusts to the city's local time
    }
    
    # Make the request
    response = requests.get(base_url, params=params)
    
    # Check if the request was successful
    if response.status_code == 200:
        weather_data[city] = response.json()
        print(f"Successfully retrieved {city} data!")
    else:
        print(f"Error fetching {city}: {response.status_code}")

print("\nPhase 2 Complete: Data is now stored in the 'weather_data' variable.")

In [ ]:
# Create a dictionary to store the finished DataFrames
dfs = {}

for city in locations.keys():
    # 1. Create the Hourly DataFrame
    df_hourly = pd.DataFrame(weather_data[city]['hourly'])
    df_hourly['time'] = pd.to_datetime(df_hourly['time'])
    
    # 2. Create a 'date' helper column for the join
    df_hourly['date'] = df_hourly['time'].dt.date

    # 3. Create the Daily DataFrame (Sunrise/Sunset)
    df_daily = pd.DataFrame(weather_data[city]['daily'])
    df_daily['date'] = pd.to_datetime(df_daily['time']).dt.date

    # 4. JOIN them together
    # This "broadcasts" the daily sunrise/sunset to every hour of that day
    df_combined = df_hourly.merge(
        df_daily[['date', 'sunrise', 'sunset']], 
        on='date', 
        how='left'
    )

    # 5. Final Cleanup
    df_combined.set_index('time', inplace=True)
    df_combined.drop(columns=['date'], inplace=True)
    
    # Store the finished product
    dfs[city] = df_combined

In [ ]:
print(dfs['Seattle'].head())

In [ ]:
# 1. Resample Hourly Precipitation to Weekly Totals
# 'W' stands for Weekly, and .sum() adds all the hourly rain for that week
seattle_weekly = dfs['Seattle']['precipitation'].resample('W').sum()
london_weekly = dfs['London']['precipitation'].resample('W').sum()

# 2. Create the Comparison Visualization
plt.figure(figsize=(14, 7))

# Plotting both lines
plt.plot(seattle_weekly.index, seattle_weekly.values, label='Seattle', color='#1f77b4', linewidth=2)
plt.plot(london_weekly.index, london_weekly.values, label='London', color='#ff7f0e', linewidth=2)

# 3. Formatting the Chart
plt.title('Weekly Precipitation Comparison: Seattle vs London (2022)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Precipitation (mm)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

# Tight layout ensures labels aren't cut off
plt.tight_layout()
plt.show()

# 4. Global Analysis Summary
print(f"Total Annual Rainfall in Seattle (2022): {seattle_weekly.sum():.2f} mm")
print(f"Total Annual Rainfall in London (2022): {london_weekly.sum():.2f} mm")

In [ ]:
#Now we play around with different differences in daylight hours
# 1. Ensure sunrise and sunset are in datetime format
for city in ['Seattle', 'London']:
    dfs[city]['sunrise'] = pd.to_datetime(dfs[city]['sunrise'])
    dfs[city]['sunset'] = pd.to_datetime(dfs[city]['sunset'])
    
    # 2. Calculate the daylight duration in hours
    # (Sunset - Sunrise) gives a Timedelta; .dt.total_seconds() / 3600 converts it to float hours
    dfs[city]['daylight_hours'] = (dfs[city]['sunset'] - dfs[city]['sunrise']).dt.total_seconds() / 3600

In [ ]:
type(dfs)

In [ ]:
dfs

In [ ]:
# 3. Resample to Monthly Averages
# 'MS' stands for Month Start, which makes the x-axis labels cleaner
seattle_daylight_monthly = dfs['Seattle']['daylight_hours'].resample('MS').mean()
london_daylight_monthly = dfs['London']['daylight_hours'].resample('MS').mean()
london_daylight_monthly

In [ ]:
# 4. Create the Visualization
plt.figure(figsize=(12, 6))

plt.plot(seattle_daylight_monthly.index, seattle_daylight_monthly.values, 
         label='Seattle', marker='o', color='#1f77b4', linewidth=2)
plt.plot(london_daylight_monthly.index, london_daylight_monthly.values, 
         label='London', marker='s', color='#ff7f0e', linewidth=2)

# Add a horizontal line at 12 hours for reference (the Equinox)
plt.axhline(y=12, color='gray', linestyle='--', alpha=0.5, label='12-hour mark')

# 5. Formatting
plt.title('Average Monthly Daylight Hours: Seattle vs London (2022)', fontsize=14)
plt.ylabel('Hours of Daylight', fontsize=12)
plt.xlabel('Month', fontsize=12)
plt.xticks(seattle_daylight_monthly.index, [d.strftime('%b') for d in seattle_daylight_monthly.index])
plt.grid(True, axis='y', linestyle=':', alpha=0.7)
plt.legend()

plt.show()

In [ ]:
#now, for a more dramatic difference
# 1. Define new coordinates
locations_new = {
    "Caracas": {"lat": 10.4806, "lon": -66.8983},
    "Edinburgh": {"lat": 55.9533, "lon": -3.1883}
}

weather_data_new = {}

# 2. Fetch the data (Reuse the same params logic from before)
for city, coords in locations_new.items():
    print(f"Fetching 2022 data for {city}...")
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": "2022-01-01",
        "end_date": "2022-12-31",
        "hourly": "temperature_2m", # We just need a placeholder for hourly
        "daily": "sunrise,sunset",
        "timezone": "auto"
    }
    r = requests.get(base_url, params=params)
    weather_data_new[city] = r.json()

print("Data retrieved.")

In [ ]:
plt.figure(figsize=(12, 6))

for city in ["Caracas", "Edinburgh"]:
    # Create temporary DataFrame
    temp_df = pd.DataFrame(weather_data_new[city]['daily'])
    temp_df['time'] = pd.to_datetime(temp_df['time'])
    
    # Calculate daylight hours
    sunrise = pd.to_datetime(temp_df['sunrise'])
    sunset = pd.to_datetime(temp_df['sunset'])
    temp_df['daylight_hours'] = (sunset - sunrise).dt.total_seconds() / 3600
    
    # Resample to Monthly Mean
    temp_df.set_index('time', inplace=True)
    monthly_avg = temp_df['daylight_hours'].resample('MS').mean()
    
    # Plot
    plt.plot(monthly_avg.index, monthly_avg.values, label=city, marker='o', linewidth=2)

# Formatting
plt.axhline(y=12, color='black', linestyle='--', alpha=0.3, label='Equator Standard (12h)')
plt.title('Extreme Latitudinal Comparison: Caracas vs. Edinburgh (2022)', fontsize=14)
plt.ylabel('Hours of Daylight', fontsize=12)
plt.xticks(monthly_avg.index, [d.strftime('%b') for d in monthly_avg.index])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()